In [74]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [75]:
df = pd.read_csv('DaftarSaham.csv')


In [76]:
df.columns


Index(['Code', 'Name', 'ListingDate', 'Shares', 'ListingBoard', 'Sector',
       'LastPrice', 'MarketCap', 'MinutesFirstAdded', 'MinutesLastUpdated',
       'HourlyFirstAdded', 'HourlyLastUpdated', 'DailyFirstAdded',
       'DailyLastUpdated'],
      dtype='object')

In [77]:
df['Sector'].unique()


array(['Consumer Non-Cyclicals', 'Consumer Cyclicals', 'Financials',
       'Industrials', 'Infrastructures', 'Properties & Real Estate',
       'Basic Materials', 'Energy', 'Transportation & Logistic',
       'Technology', 'Healthcare'], dtype=object)

In [78]:
df_financial = df[df['Sector'] == 'Financials']
df_financial


,Code,Name,ListingDate,Shares,ListingBoard,Sector,LastPrice,MarketCap,MinutesFirstAdded,MinutesLastUpdated,HourlyFirstAdded,HourlyLastUpdated,DailyFirstAdded,DailyLastUpdated
2,ABDA,Asuransi Bina Dana Arta Tbk.,1989-07-06,6.208067e+08,Pengembangan,Financials,6700.0,4.159405e+12,2021-11-01 09:00:00,2022-11-11 15:59:00,2020-04-16 09:00:00,2022-11-11 16:00:00,2001-04-16,2023-01-06
9,ADMF,Adira Dinamika Multi Finance T,2004-03-31,1.000000e+09,Utama,Financials,8850.0,8.850000e+12,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2004-03-31,2023-01-06
15,AGRO,Bank Raya Indonesia Tbk.,2003-08-08,2.252005e+10,Utama,Financials,406.0,9.143142e+12,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2003-08-08,2023-01-06
16,AGRS,Bank IBK Indonesia Tbk.,2014-12-22,1.748160e+10,Pengembangan,Financials,89.0,1.555863e+12,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2014-12-22,2023-01-06
17,AHAP,Asuransi Harta Aman Pratama Tb,1990-09-14,2.940000e+09,Pengembangan,Financials,64.0,1.881600e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2001-04-16,2023-01-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,VINS,Victoria Insurance Tbk.,2015-09-28,1.460574e+09,Pengembangan,Financials,133.0,1.942563e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2015-09-28,2023-01-06
800,VRNA,Verena Multi Finance Tbk.,2008-06-25,5.687354e+09,Pengembangan,Financials,99.0,5.630480e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2008-06-25,2023-01-06
801,VTNY,Venteny Fortuna International,2022-12-15 00:00:00,6.265193e+09,Pengembangan,Financials,444.0,2.781746e+12,2022-12-15 09:00:00,2023-01-06 15:59:00,2022-12-15 09:00:00,2023-01-06 15:00:00,2022-12-15,2023-01-06
815,WOMF,Wahana Ottomitra Multiartha Tb,2004-12-13,3.481481e+09,Utama,Financials,256.0,8.912593e+11,2021-11-01 09:00:00,2023-01-06 15:59:00,2020-04-16 09:00:00,2023-01-06 15:00:00,2004-12-13,2023-01-06


In [79]:
df_financial.shape

(106, 14)

In [80]:
import os

folder_path = "daily"

all_files = os.listdir(folder_path)


In [81]:
financial_codes = df_financial['Code'].tolist()

In [82]:
financial_files = [
    f for f in all_files 
    if f.replace(".csv", "") in financial_codes
]


In [83]:
data_financial = []

for file in financial_files:
    path = os.path.join(folder_path, file)
    df_temp = pd.read_csv(path)
    df_temp['Code'] = file.replace(".csv", "")
    data_financial.append(df_temp)

data_financial = pd.concat(data_financial, ignore_index=True)


In [84]:
data_financial


,timestamp,open,low,high,close,volume,Code
0,2001-04-16,194,152,194,152,0,ABDA
1,2001-04-17,194,152,194,152,0,ABDA
2,2001-04-18,194,152,194,152,0,ABDA
3,2001-04-19,194,152,194,152,0,ABDA
4,2001-04-20,194,152,194,152,0,ABDA
...,...,...,...,...,...,...,...
402615,2023-01-02,2230,2230,2230,2230,0,YULE
402616,2023-01-03,2230,2230,2230,2230,0,YULE
402617,2023-01-04,2230,2230,2230,2230,500,YULE
402618,2023-01-05,2230,2230,2230,2230,0,YULE


In [85]:
H=14

In [86]:
data_financial["future_close"] = data_financial["close"].shift(-H)
data_financial["future_return"] = (data_financial["future_close"] - data_financial["close"]) / data_financial["close"]


In [ ]:
buy_threshold = 0.03   # +3% over 14 days
sell_threshold = -0.03 # -3% over 14 days

data_financial["label"] = 0 # hold

data_financial.loc[data_financial["future_return"] > buy_threshold, "label"] = 1 # buy
data_financial.loc[data_financial["future_return"] < sell_threshold, "label"] = -1 # sell

In [88]:
data_financial = data_financial.dropna()

In [91]:
data_financial.head(500)

,timestamp,open,low,high,close,volume,Code,future_close,future_return,label
0,2001-04-16,194,152,194,152,0,ABDA,152.0,0.000000,0
1,2001-04-17,194,152,194,152,0,ABDA,152.0,0.000000,0
2,2001-04-18,194,152,194,152,0,ABDA,152.0,0.000000,0
3,2001-04-19,194,152,194,152,0,ABDA,152.0,0.000000,0
4,2001-04-20,194,152,194,152,0,ABDA,152.0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...
495,2003-03-10,124,124,124,124,0,ABDA,124.0,0.000000,0
496,2003-03-11,124,124,124,124,0,ABDA,124.0,0.000000,0
497,2003-03-12,124,124,124,124,0,ABDA,124.0,0.000000,0
498,2003-03-13,124,124,124,124,0,ABDA,124.0,0.000000,0
